# 🔎 Visor de EEG · ver el EEG de un paciente

> 🧭 **En 30 segundos**
> - **Para qué:** elegir un paciente de TUSZ y **ver su EEG** como en un electroencefalógrafo, con las **crisis sombreadas**.
> - **Entra:** el inventario de **F01.01** (`edfs.csv`, para la lista de pacientes) y los `.edf` / `.csv_bi` del paciente → **Sale:** nada, solo se mira.
> - **Idea clave:** es una herramienta, no una fase. No guarda ficheros ni cambia nada.

## Cómo se usa

| Control | Qué hace |
|---|---|
| **Partición** | 🔵 train, 🟡 dev o 🟢 eval. Con *Solo pacientes con crisis* se ocultan los que no tienen ninguna |
| **Paciente** | Ordenados de más a menos crisis, con sus tipos (🟥 fnsz (♦️), 🟦 gnsz (🔷)...) |
| **Fichero** | Los EDF del paciente: sesión, montaje, duración y crisis. Al elegirlo salta a la primera crisis. Solo se lee del disco **la ventana que se ve**, así que va igual de rápido con ficheros de horas |
| **Ir a crisis** | Salta a 5 s antes de la crisis elegida |
| **◀ Inicio ▶** | Moverse por el fichero: el deslizador o una ventana entera atrás/adelante |
| **Ventana / Escala** | Segundos que se ven a la vez y µV entre canales (más pequeño = ondas más grandes) |
| **Filtro** | El mismo del TFM (`senal/filtrado.py`): paso banda 0,5-40 Hz + notch 60 Hz. Sin él se ve la señal cruda |
| **📊 Frecuencias** | Debajo del trazado: el **espectro** de la ventana (cuánta potencia hay en cada frecuencia), el **ritmo dominante** en Hz y **una cabeza por banda** (δ, θ, α, β, γ): en qué electrodos pesa más cada banda |
| **🧠 ¿Dónde se origina?** | Para la crisis elegida (o la que se ve): compara sus **5 primeros segundos** con una **señal basal** sin crisis de antes. Enseña al momento una **cabeza** con cuánto crece la potencia en cada electrodo y, como en **F01.03v**, el **cerebro 3D** con el origen estimado (sLORETA) y el lóbulo |

**Términos:**
- **Espectro**: descompone la señal en frecuencias, como un ecualizador. Las bandas: **delta** (0,5-4 Hz, lentas), **theta** (4-8), **alfa** (8-13), **beta** (13-30) y **gamma** (30-45, rápidas).
- **Ritmo dominante**: la frecuencia que más **destaca**. El EEG siempre tiene más potencia en lo lento (cae como 1/f), así que se busca quitando esa caída.
- **dB (decibelios)**: +10 dB = 10 veces más potencia que en el basal; 0 dB = igual; negativo = menos.
- **sLORETA**: método que estima, a partir de los electrodos, qué punto **dentro** del cerebro genera la actividad. Usa un cerebro plantilla y ~19 electrodos: precisión de **lóbulo**, orientativo, no diagnóstico.
- **Montaje TCP**: cada canal es la **resta de dos electrodos vecinos** (p. ej. `FP1-F7`). Son 22 canales en 5 cadenas, separadas en la figura por **líneas rojas finas**: temporal izquierda, temporal derecha, central, parasagital izquierda y parasagital derecha.
- **Barra roja en L** (abajo a la derecha): mide **la escala en µV** (alto) y **1 segundo** (ancho), para leer amplitudes y tiempos.
- **Zona sombreada**: crisis según el `.csv_bi`; el color y el texto son su **tipo dominante** (`crisis_del_fichero()` de `dataset/anotaciones.py`).

> 💡 Al principio de muchos ficheros de TUSZ la señal es **plana unos segundos** (el equipo aún no grababa). El salto al empezar la señal real deja un pequeño eco del filtro: no es parte del EEG.

## 1. Librerías

Primero se comprueba que están las librerías (y se instalan solas si falta alguna). El visor está en `tfm_comun/visual/visor_eeg.py`; el dibujo, en `visual/trazado_eeg.py`; la lista de pacientes, en `dataset/catalogo.py`.

In [1]:
# Recarga automática: si cambia un módulo de tfm_comun, se vuelve a cargar solo (sin reiniciar el kernel)
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

# tfm_comun está en ALMA, por encima de la carpeta de este notebook: lo añadimos al path de Python
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tfm_comun').is_dir())))
# Vuelve a leer tfm_comun del disco cada vez que se ejecuta esta celda (por si ha cambiado), sin reiniciar el kernel
for modulo in [m for m in list(sys.modules) if m == 'tfm_comun' or m.startswith('tfm_comun.')]:
    del sys.modules[modulo]

from tfm_comun.entorno.requisitos import REQUISITOS_VISOR, comprobar_requisitos

disponibles = comprobar_requisitos(REQUISITOS_VISOR)

,estado,versión,para qué
librería,,,
numpy,✅,2.5.4,cálculo con matrices
pandas,✅,3.0.6,tablas (inventario de F01.01 y anotaciones)
matplotlib,✅,3.11.2,dibujar el EEG
scipy,✅,1.18.1,filtro paso banda y notch
mne,✅,1.13.2,leer el EDF y montaje TCP
ipywidgets,✅,8.1.9,"los controles del visor (desplegables, deslizador, botones)"
pyvista,✅,0.49.2,"el cerebro 3D del botón 🧠 (sin él, solo la cabeza)"


Todo lo necesario está instalado.


In [2]:
import ipywidgets
import matplotlib
import mne
import numpy as np
import pandas as pd

from tfm_comun.entorno.versiones import mostrar_versiones
from tfm_comun.visual.visor_eeg import VisorEEG

versiones_usadas = mostrar_versiones(np, pd, matplotlib, mne, ipywidgets)

Python      3.14.0
numpy       2.5.4
pandas      3.0.6
matplotlib  3.11.2
mne         1.13.2
ipywidgets  8.1.9


## 2. Visor

Elige partición, paciente y fichero. Cada cambio lee del disco solo la ventana que se ve (menos de medio segundo). El botón 🧠 tarda más la primera vez (prepara el modelo de la cabeza); después se reutiliza.

In [3]:
VisorEEG().mostrar()